In [1]:
!pip install wfdb

In [2]:
import wfdb
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split

def load_and_segment_data(record_list, data_path='/home/arnab/Downloads/physionet.org/files/mitdb/1.0.0/', window_size=256):
   
    X = []
    y = []
    half_window = window_size // 2

    for record_name in record_list:
        print(f"Processing record: {record_name}")
        # Read the signal and annotations
        record = wfdb.rdrecord(data_path + str(record_name))
        annotation = wfdb.rdann(data_path + str(record_name), 'atr')
        
        # Use the first channel (usually MLII - modified lead II)
        signal = record.p_signal[:, 0] 
        symbols = annotation.symbol
        sample_indices = annotation.sample
        
        for i in range(len(symbols)):
            sym = symbols[i]
            idx = sample_indices[i]
            
            # Skip if the window falls outside the signal boundaries
            if idx - half_window < 0 or idx + half_window >= len(signal):
                continue

            label = 0 if sym == 'N' else 1
            
            # Extract the 256-sample window
            window = signal[idx - half_window : idx + half_window]
            
            X.append(window)
            y.append(label)

    X = np.array(X)[..., np.newaxis] 
    y = np.array(y)
    return X, y

def build_fpga_cnn(input_shape):Scale Factor: 176.9059)
  -> Saved fpga_weights/layer_2_conv1d_1_biases.coe (Scale Factor: 696.9517)
Processing Layer 5: dense
  -> Saved fpga_weights/layer_5_dense_weights.coe (Scale Factor: 175.4038)
  -> Saved fpga_weights/layer_5_dense_biases.coe (Scale Factor: 2513.1917)
Processing Layer 6: dense_1
  -> Saved fpga_weights/layer_6_dense_1_weights.coe (Scale Factor: 161.8447)
  -> Saved fpga_weights/layer_6_dense_1_biases.coe (Scale Factor: 537.6711)
    """
    A drastically shrunken 1D-CNN. 
    
    """
    model = models.Sequential([
        layers.InputLayer(input_shape=input_shape),
        
        layers.Conv1D(filters=8, kernel_size=5, activation='relu'),
        layers.MaxPooling1D(pool_size=2), # Shrinks data size by half
        
        layers.Conv1D(filters=16, kernel_size=3, activation='relu'),
        layers.MaxPooling1D(pool_size=2),
        
        layers.Flatten(),
        
        layers.Dense(16, activation='relu'),
        
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam', 
                  loss='binary_crossentropy', 
                  metrics=['accuracy'])
    return model


if __name__ == "__main__":
    records = ['100', '101', '105', '106', '108', '109', '111', '112', '113', '114']
    
    print("Extracting data...")
    X, y = load_and_segment_data(records, window_size=256)
    print(f"Total heartbeats extracted: {len(X)}")
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    print("Building model...")
    model = build_fpga_cnn(input_shape=(256, 1))
    model.summary() 
    
    print("Training model...")
    history = model.fit(X_train, y_train, 
                        epochs=10, 
                        batch_size=64, 
                        validation_data=(X_test, y_test))
    
    model.save("ecg_fpga_base_model.h5")
    print("Training complete. Model saved.")

2026-03-17 02:34:07.166381: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Extracting data...
Processing record: 100
Processing record: 101
Processing record: 105
Processing record: 106
Processing record: 108
Processing record: 109
Processing record: 111
Processing record: 112
Processing record: 113
Processing record: 114
Total heartbeats extracted: 21646
Building model...


/home/arnab/Downloads/python venv/venv/lib/python3.12/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
I0000 00:00:1773695052.472584   22339 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2158 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 252, 8)         │            48 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 126, 8)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 124, 16)        │           400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 62, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 992)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │        15,888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

...

2026-03-17 02:34:16.787177: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_351', 12 bytes spill stores, 12 bytes spill loads

2026-03-17 02:34:16.854144: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_351', 12 bytes spill stores, 12 bytes spill loads



271/271 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8796 - loss: 0.3120

2026-03-17 02:34:18.720452: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_85', 12 bytes spill stores, 12 bytes spill loads

2026-03-17 02:34:18.720480: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_85', 12 bytes spill stores, 12 bytes spill loads



271/271 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.9368 - loss: 0.1964 - val_accuracy: 0.9704 - val_loss: 0.1200
Epoch 2/10
271/271 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9734 - loss: 0.0931 - val_accuracy: 0.9734 - val_loss: 0.0944
Epoch 3/10
271/271 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9781 - loss: 0.0748 - val_accuracy: 0.9737 - val_loss: 0.0862
Epoch 4/10
271/271 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9806 - loss: 0.0628 - val_accuracy: 0.9755 - val_loss: 0.0762
Epoch 5/10
271/271 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9833 - loss: 0.0530 - val_accuracy: 0.9771 - val_loss: 0.0749
Epoch 6/10
271/271 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9863 - loss: 0.0467 - val_accuracy: 0.9794 - val_loss: 0.0746
Epoch 7/10
271/271 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9868 - loss: 0.0411 - val_accuracy: 0.9806 - val_loss: 0.0689
Epoch 8/10
271/271 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9879 - loss: 0.0389 - val_accuracy: 0.9822 - val

Training complete. Model saved.


In [3]:
import tensorflow as tf
import numpy as np
import os

# 1. Load your trained model
model = tf.keras.models.load_model("ecg_fpga_base_model.h5")

# Create a directory to store the COE files
output_dir = "fpga_weights"
os.makedirs(output_dir, exist_ok=True)

def write_coe_file(filename, data_array):
    """Writes a 1D numpy array of integers to a Xilinx .coe file."""
    with open(filename, 'w') as f:
        f.write("memory_initialization_radix=10;\n")
        f.write("memory_initialization_vector=\n")
        
        # Flatten the array and convert to list of strings
        flat_data = data_array.flatten()
        data_strs = [str(int(val)) for val in flat_data]
        
        # Write comma-separated values, ending with a semicolon
        f.write(",\n".join(data_strs))
        f.write(";\n")

for i, layer in enumerate(model.layers):
    weights_biases = layer.get_weights()
    
    if len(weights_biases) > 0:
        print(f"Processing Layer {i}: {layer.name}")
        weights = weights_biases[0]
        biases = weights_biases[1] if len(weights_biases) > 1 else None

        # --- QUANTIZE WEIGHTS ---
        max_w = np.max(np.abs(weights))
        scale_w = 127.0 / max_w if max_w != 0 else 1.0
        quantized_weights = np.round(weights * scale_w).astype(np.int8)
        
        # Save Weights to COE
        w_filename = os.path.join(output_dir, f"layer_{i}_{layer.name}_weights.coe")
        write_coe_file(w_filename, quantized_weights)
        print(f"  -> Saved {w_filename} (Scale Factor: {scale_w:.4f})")

        if biases is not None:
            max_b = np.max(np.abs(biases))
            scale_b = 127.0 / max_b if max_b != 0 else 1.0
            quantized_biases = np.round(biases * scale_b).astype(np.int8)
            
            # Save Biases to COE
            b_filename = os.path.join(output_dir, f"layer_{i}_{layer.name}_biases.coe")
            write_coe_file(b_filename, quantized_biases)
            print(f"  -> Saved {b_filename} (Scale Factor: {scale_b:.4f})")

print("\nSuccess! All weights and biases are ready for Vivado.")

Processing Layer 0: conv1d
  -> Saved fpga_weights/layer_0_conv1d_weights.coe (Scale Factor: 206.8567)
  -> Saved fpga_weights/layer_0_conv1d_biases.coe (Scale Factor: 402.1466)
Processing Layer 2: conv1d_1
  -> Saved fpga_weights/layer_2_conv1d_1_weights.coe (Scale Factor: 176.9059)
  -> Saved fpga_weights/layer_2_conv1d_1_biases.coe (Scale Factor: 696.9517)
Processing Layer 5: dense
  -> Saved fpga_weights/layer_5_dense_weights.coe (Scale Factor: 175.4038)
  -> Saved fpga_weights/layer_5_dense_biases.coe (Scale Factor: 2513.1917)
Processing Layer 6: dense_1
  -> Saved fpga_weights/layer_6_dense_1_weights.coe (Scale Factor: 161.8447)
  -> Saved fpga_weights/layer_6_dense_1_biases.coe (Scale Factor: 537.6711)

Success! All weights and biases are ready for Vivado.


In [1]:
import wfdb
import numpy as np
from tensorflow.keras.models import load_model

def load_and_segment_data(record_list, data_path='/home/arnab/Downloads/physionet.org/files/mitdb/1.0.0/', window_size=256):
    """
    Reads WFDB files, finds R-peaks, and extracts a window of samples.
    Labels: 0 for Normal ('N'), 1 for Abnormal (Arrhythmia).
    """
    X = []
    y = []
    half_window = window_size // 2

    for record_name in record_list:
        print(f"Processing record: {record_name}")
        record = wfdb.rdrecord(data_path + str(record_name))
        annotation = wfdb.rdann(data_path + str(record_name), 'atr')
        
        signal = record.p_signal[:, 0] 
        symbols = annotation.symbol
        sample_indices = annotation.sample
        
        for i in range(len(symbols)):
            sym = symbols[i]
            idx = sample_indices[i]
            
            if idx - half_window < 0 or idx + half_window >= len(signal):
                continue
            
            label = 0 if sym == 'N' else 1
            window = signal[idx - half_window : idx + half_window]
            
            X.append(window)
            y.append(label)

    X = np.array(X)[..., np.newaxis] 
    y = np.array(y)
    return X, y

print("\nLoading frozen model...")
model = load_model('ecg_fpga_base_model.h5')

records_to_search = ['100', '101', '105', '106'] 
data_path = '/home/arnab/Downloads/physionet.org/files/mitdb/1.0.0/' # Change if needed!

print("Extracting raw patient data...")
X_raw, y_true = load_and_segment_data(records_to_search, data_path=data_path, window_size=256)

print("\nAsking model to diagnose patients...")
predictions = model.predict(X_raw)

sick_indices = np.where((predictions > 0.8) & (y_true == 1))[0]

if len(sick_indices) == 0:
    print("Uh oh, no high-confidence sick patients found. Try adding more records to 'records_to_search'!")
else:
    sick_patient_idx = sick_indices[0] # Grab the first one
    print(f"\n--- SUCCESS ---")
    print(f"Found a Class 1 (Arrhythmia) patient at index: {sick_patient_idx}")
    print(f"Model's confidence: {predictions[sick_patient_idx][0] * 100:.2f}%")

    # Extract their 256 data points
    raw_data = X_raw[sick_patient_idx].flatten()


    QUANT_MULTIPLIER = 127.0 
    
    quantized_data = np.clip(np.round(raw_data * QUANT_MULTIPLIER), -128, 127).astype(int)

    # Convert to 8-bit Two's Complement Hex
    hex_strings = []
    for val in quantized_data:
        hex_val = "{:02X}".format(int(val) & 0xFF)
        hex_strings.append(hex_val)

    # Write to file
    with open("sick_patient_golden.txt", "w") as f:
        for i in range(0, 256, 16):
            row = " ".join(hex_strings[i:i+16])
            f.write(row + "\n")
            
    print("\nsick_patient_golden.txt generated successfully! Drop this in Vivado.")

2026-03-29 02:55:58.930769: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.



Loading frozen model...


I0000 00:00:1774733162.592220   88796 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1959 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Extracting raw patient data...
Processing record: 100
Processing record: 101
Processing record: 105
Processing record: 106

Asking model to diagnose patients...


2026-03-29 02:56:03.664370: I external/local_xla/xla/service/service.cc:163] XLA service 0x793cb8005530 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-29 02:56:03.664390: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6
2026-03-29 02:56:03.681572: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-03-29 02:56:03.727203: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91301


240/280 ━━━━━━━━━━━━━━━━━━━━ 0s 632us/step

I0000 00:00:1774733164.898327  107110 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


280/280 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step

--- SUCCESS ---
Found a Class 1 (Arrhythmia) patient at index: 1905
Model's confidence: 99.99%

sick_patient_golden.txt generated successfully! Drop this in Vivado.


In [4]:
import wfdb
import numpy as np
import time
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

def load_and_segment_data(record_list, data_path='/home/arnab/Downloads/physionet.org/files/mitdb/1.0.0/', window_size=256):
    X = []
    y = []
    half_window = window_size // 2

    for record_name in record_list:
        record = wfdb.rdrecord(data_path + str(record_name))
        annotation = wfdb.rdann(data_path + str(record_name), 'atr')
        
        signal = record.p_signal[:, 0] 
        symbols = annotation.symbol
        sample_indices = annotation.sample
        
        for i in range(len(symbols)):
            sym = symbols[i]
            idx = sample_indices[i]
            if idx - half_window < 0 or idx + half_window >= len(signal): continue
            label = 0 if sym == 'N' else 1
            window = signal[idx - half_window : idx + half_window]
            X.append(window)
            y.append(label)

    X = np.array(X)[..., np.newaxis] 
    y = np.array(y)
    return X, y


records = ['100', '101', '105', '106', '108', '109', '111', '112', '113', '114']
data_path = '/home/arnab/Downloads/physionet.org/files/mitdb/1.0.0/' # Change if needed!

print("Extracting data from PhysioNet... (This might take a few seconds)")
X, y = load_and_segment_data(records, data_path=data_path, window_size=256)

print(f"\n---> REPORT SECTION 4: Total Samples Extracted = {len(X)} <---")

# Recreate the exact 80/20 split so the test data matches your original run perfectly
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = load_model('ecg_fpga_base_model.h5')

print("\n--- REPORT SECTION 6: PRECISION, RECALL, & F1-SCORE ---")
# Get probabilities and convert to binary predictions
y_pred_probs = model.predict(X_test, verbose=0)
y_pred_binary = (y_pred_probs > 0.5).astype(int)

# Print the beautiful sklearn classification report
report = classification_report(y_test, y_pred_binary, target_names=['Normal (0)', 'Arrhythmia (1)'])
print(report)

print("\n--- REPORT SECTION 6: CPU INFERENCE LATENCY ---")
# Grab ONE patient to simulate real-time edge inference
single_patient = np.expand_dims(X_test[0], axis=0)

_ = model.predict(single_patient, verbose=0)

# Time the actual inference
start_time = time.time()
_ = model.predict(single_patient, verbose=0)
end_time = time.time()

cpu_latency_ms = (end_time - start_time) * 1000
print(f"Single Patient CPU Latency: {cpu_latency_ms:.2f} milliseconds")

Extracting data from PhysioNet... (This might take a few seconds)



---> REPORT SECTION 4: Total Samples Extracted = 21646 <---

--- REPORT SECTION 6: PRECISION, RECALL, & F1-SCORE ---
                precision    recall  f1-score   support

    Normal (0)       0.98      1.00      0.99      3195
Arrhythmia (1)       0.99      0.95      0.97      1135

      accuracy                           0.98      4330
     macro avg       0.98      0.97      0.98      4330
  weighted avg       0.98      0.98      0.98      4330


--- REPORT SECTION 6: CPU INFERENCE LATENCY ---
Single Patient CPU Latency: 49.51 milliseconds
